# Fraud Detection - EDA & Data Quality Checks
This notebook performs Exploratory Data Analysis on `raw_insertions.parquet`.
Goals:
1. Check data quality (nulls, types).
2. Analyze field coverage.
3. Explore fraud distribution and rates.
4. Analyze fraud label maturity (delay).

In [6]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

# Configure Logic (path relative to notebooks/ directory)
DATA_PATH = "../artifacts/raw_insertions.parquet"

## 1. Load Data & Basic Info

In [7]:
print(f"Loading data from {DATA_PATH}...")
df = pl.read_parquet(DATA_PATH)
print(f"Shape: {df.shape}")
print("-" * 50)
print("Schema:")
print(df.schema)

Loading data from ../artifacts/raw_insertions.parquet...
Shape: (234458, 292)
--------------------------------------------------
Schema:
Schema({'account_created_at': Datetime(time_unit='us', time_zone='UTC'), 'auto_approval_criteria.criteria': Null, 'auto_approval_criteria.criteria.hasUsedSameEmailBefore': Boolean, 'auto_approval_criteria.criteria.seonApproved': Boolean, 'auto_approval_criteria.criteria.swissPostAddressValidation': String, 'auto_approval_criteria.meta': Null, 'auto_approval_criteria.meta.seonFraudScore': Float64, 'auto_approval_criteria.meta.state': String, 'auto_approval_criteria.seonSession': String, 'bundle.id': String, 'bundle.immoScout24UpsellPrice': Float64, 'bundle.initialPeriodPrice': Float64, 'bundle.initialPrice': Float64, 'bundle.oldInitialPrice': Float64, 'bundle.period': Int64, 'bundle.preselectionAddOnPrice': Float64, 'bundle.recurringPeriodPrice': Float64, 'bundle.recurringPrice': Float64, 'bundle.tier': String, 'bundle.voluntaryTenantPlusSelected': Boo

## 2. Data Quality & Coverage

In [8]:
# Check Nulls
null_counts = df.null_count()
total_rows = df.height

print("\nMissing Values & Coverage:")
coverage_stats = []
for col in df.columns:
    nulls = null_counts[col][0]
    pct_missing = (nulls / total_rows) * 100
    coverage = 100 - pct_missing
    dtype = str(df.schema[col])
    
    coverage_stats.append({
        "column": col,
        "dtype": dtype,
        "nulls": nulls,
        "pct_missing": round(pct_missing, 2),
        "coverage": round(coverage, 2)
    })

# Convert to simple dataframe for display
df_coverage = pl.DataFrame(coverage_stats).sort("pct_missing", descending=True)
print(df_coverage)

# Check Empty Strings (for string cols)
print("\nEmpty String Counts (for string columns):")
for col in df.select(pl.col(pl.Utf8)).columns:
    empty_count = df.filter(pl.col(col) == "").height
    if empty_count > 0:
        print(f"{col}: {empty_count} ({round(empty_count/total_rows*100, 2)}%)")


Missing Values & Coverage:
shape: (292, 5)
┌─────────────────────────────────┬──────────────────────────┬────────┬─────────────┬──────────┐
│ column                          ┆ dtype                    ┆ nulls  ┆ pct_missing ┆ coverage │
│ ---                             ┆ ---                      ┆ ---    ┆ ---         ┆ ---      │
│ str                             ┆ str                      ┆ i64    ┆ f64         ┆ f64      │
╞═════════════════════════════════╪══════════════════════════╪════════╪═════════════╪══════════╡
│ auto_approval_criteria.criteri… ┆ Null                     ┆ 234458 ┆ 100.0       ┆ 0.0      │
│ auto_approval_criteria.meta     ┆ Null                     ┆ 234458 ┆ 100.0       ┆ 0.0      │
│ bundle.immoScout24UpsellPrice   ┆ Float64                  ┆ 234456 ┆ 100.0       ┆ 0.0      │
│ listing.address.geoCoordinates… ┆ Boolean                  ┆ 234453 ┆ 100.0       ┆ 0.0      │
│ listing.address.geoDistances    ┆ String                   ┆ 234453 ┆ 100.0      

## 3. Type Distribution & Unique Values

In [9]:
print("\nUnique Value Counts:")
for col in df.columns:
    # Cap generic printing
    n_unique = df.select(pl.col(col).n_unique()).item()
    print(f"{col}: {n_unique} unique")


Unique Value Counts:
account_created_at: 125734 unique
auto_approval_criteria.criteria: 1 unique
auto_approval_criteria.criteria.hasUsedSameEmailBefore: 3 unique
auto_approval_criteria.criteria.seonApproved: 3 unique
auto_approval_criteria.criteria.swissPostAddressValidation: 6 unique
auto_approval_criteria.meta: 1 unique
auto_approval_criteria.meta.seonFraudScore: 5104 unique
auto_approval_criteria.meta.state: 4 unique
auto_approval_criteria.seonSession: 176593 unique
bundle.id: 7884 unique
bundle.immoScout24UpsellPrice: 2 unique
bundle.initialPeriodPrice: 81 unique
bundle.initialPrice: 537 unique
bundle.oldInitialPrice: 204 unique
bundle.period: 5 unique
bundle.preselectionAddOnPrice: 3 unique
bundle.recurringPeriodPrice: 80 unique
bundle.recurringPrice: 541 unique
bundle.tier: 4 unique
bundle.voluntaryTenantPlusSelected: 2 unique
contact_emails_hash: 114282 unique
customer_segment: 5 unique
first_published_date: 212135 unique
fraud_flag: 19218 unique
listing.address.address_hash: 1

## 4. Fraud Distribution

In [10]:
# Target distribution
# Assuming 'is_fraud' is the target. If not, we look for status transitions or similar.
# The user mentioned 'raw_insertions'. Does it have labels?
# Typically 'raw_insertions' might need joining with 'status_history'.
# Let's check if 'is_fraud' exists. Not in schema?
# If missing, we might need to derive it or finding 'state' column.
# Let's inspect columns to find target.
# Look for specific columns based on previous run
TARGET_COL = "fraud_flag" 
time_col = "submission_at"

if TARGET_COL not in df.columns:
    print(f"Target '{TARGET_COL}' still not found. Available columns with 'fraud':")
    print([c for c in df.columns if "fraud" in c.lower()])
else:
    print(f"\nFraud Rate ({TARGET_COL}):")
    # Check values
    print(df[TARGET_COL].value_counts())
    
    # Check if boolean or int or string
    dtypes = df.schema[TARGET_COL]
    print(f"Target Type: {dtypes}")

# Time Analysis
if time_col in df.columns:
    # Ensure datetime
    # df = df.with_columns(pl.col(time_col).str.to_datetime()) # It might be string?
    
    min_date = df[time_col].min()
    max_date = df[time_col].max()
    print(f"\nTime Range: {min_date} to {max_date}")
    
    # Analyze daily volume
    # Cast to date if needed
    daily_vol = (
        df.group_by(pl.col(time_col).dt.date())
        .count()
        .sort(time_col)
    )
    
    # Plotting code remains similar...
    
# Feature Discovery (Bedrooms, Price, Area)
print("\nFeature Discovery:")
keywords = ["price", "area", "room", "bed", "bath", "m2", "size"]
for k in keywords:
    matches = [c for c in df.columns if k in c.lower()]
    print(f"Columns matching '{k}': {len(matches)}")
    if len(matches) < 10:
        print(matches)
    else:
        print(matches[:5] + ["..."])


Fraud Rate (fraud_flag):
shape: (19_218, 2)
┌─────────────────────────────┬───────┐
│ fraud_flag                  ┆ count │
│ ---                         ┆ ---   │
│ datetime[μs, UTC]           ┆ u32   │
╞═════════════════════════════╪═══════╡
│ 2024-09-14 18:50:20.757 UTC ┆ 1     │
│ 2024-03-01 09:04:24.423 UTC ┆ 1     │
│ 2025-05-20 06:32:32.223 UTC ┆ 1     │
│ 2024-10-04 16:07:59 UTC     ┆ 1     │
│ 2025-02-12 06:52:09.653 UTC ┆ 1     │
│ …                           ┆ …     │
│ 2023-12-08 18:10:11.632 UTC ┆ 1     │
│ 2023-10-23 14:36:45.935 UTC ┆ 1     │
│ 2023-03-10 22:31:44.654 UTC ┆ 1     │
│ 2023-02-18 17:13:20.192 UTC ┆ 1     │
│ 2023-01-27 12:16:03.577 UTC ┆ 1     │
└─────────────────────────────┴───────┘
Target Type: Datetime(time_unit='us', time_zone='UTC')

Time Range: 2023-01-01 07:36:27.382000+00:00 to 2025-11-01 23:43:13.152000+00:00

Feature Discovery:
Columns matching 'price': 20
['bundle.immoScout24UpsellPrice', 'bundle.initialPeriodPrice', 'bundle.initialPrice', 'bu

/var/folders/4m/g11mmd5d45592_f6ntg98gr80000gn/T/ipykernel_53594/1013572731.py:37: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()
